In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [1]:
!pip install -U nest-asyncio==1.6.0 pyngrok==7.2.4 uvicorn==0.34.2 fastapi==0.115.12

In [2]:
!pip install -U pandas==2.2.2 numpy==2.0.2 scipy==1.14.1 accelerate==1.6.0 peft==0.15.2 bitsandbytes==0.45.5 transformers==4.51.3 trl==0.16.1

In [3]:
# 비동기 웹 라이브러리
# FastAPI 앱 실행 위한 서버
import uvicorn

In [4]:
# API 서버 생성을 위한 핵심 라이브러리
from fastapi import FastAPI

In [5]:
# API 통신을 위한 데이터 구조 설계 라이브러리
from pydantic import BaseModel

# 데이터 처리 관련 라이브러리
import pandas as pa
import numpy as np

# 서로 다른 서버를 서로 허용하기 위한!!
from fastapi.middleware.cors import CORSMiddleware

In [6]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging
)
from transformers import AutoConfig,AutoModel
import torch
from peft import PeftModel, PeftConfig

In [7]:
# 분석 서버 앱 생성
app = FastAPI(title = "LLM API")

# 모든 출처(origin)에서의 접근을 허용하기 위한 설정
origins = ["*"]

# CORS(Cross-Origin Resource Sharing) 미들웨어 추가
# 모든 도메인, 쿠키,인증정보, HTTP 메소드, HTTP 헤더 허용
app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["GET", "POST", "PUT", "DELETE"],
    allow_headers=origins,
)


In [8]:
base_model = "yunhwa/ai_question"
### 베이스모델 불러오기
baseModel = AutoModelForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float32,
    #device_map= "auto" # T4 GPU 사용 시
    device_map={"": 0} # L4 이상 GRU 사용시
)

### 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
DEFAULT_SYSTEM_MESSAGE = "당신은 문제를 정확하게 답변하는 AI입니다."
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
# 시스템 메시지 + 문제(질문) 기반 답변 생성 함수

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    
def generate_gemma_answer(model, tokenizer, user_message, system_message=DEFAULT_SYSTEM_MESSAGE, device=DEVICE,
                          max_new_tokens=512, temperature=0.1, top_p=0.95, top_k=50):

     # Gemma 종료 토큰 둘 다 포함
    end_of_turn_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    eos_ids = list({tokenizer.eos_token_id, end_of_turn_id})  # 중복 제거

    if system_message:
        prompt = (
            f"<start_of_turn>system\n{system_message}<end_of_turn>\n"
            f"<start_of_turn>user\n{user_message}<end_of_turn>\n"
            f"<start_of_turn>model\n"
        )
    else:
        prompt = (
            f"<start_of_turn>user\n{user_message}<end_of_turn>\n"
            f"<start_of_turn>model\n"
        )
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    input_len = inputs["input_ids"].shape[1]
    model.to(device)
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )
    # 전체 디코딩
    # return tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 답변 토큰만 디코딩
    answer_tokens = outputs[0][input_len:]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()


In [10]:
# 문제(질문) 예시
user_message = (
    "AI와 머신런닝의 차이 설명해줘"
)
# 시스템 메시지(옵션)
system_message = DEFAULT_SYSTEM_MESSAGE
# 답변 생성
output = generate_gemma_answer(baseModel, tokenizer, user_message, system_message)
print("\n[질문]\n", user_message)
print("\n[답변]\n", output)


[질문]
 AI와 머신런닝의 차이 설명해줘

[답변]
 AI는 모든 똑똑한 로봇 기술이고 머신러닝은 데이터로 배우는 방법이야


In [11]:
class InDataset(BaseModel):
    question : str

In [12]:
@app.post("/generate", status_code=200)
async def predict(inQustion: InDataset):
    # inQustion = InDataset(question="스마트금융과는 어디에 위치하나요?")
    myQuestion = inQustion.question
    result = generate_gemma_answer(baseModel, tokenizer, myQuestion, system_message)
    finalResult = {"predict":result}
    return finalResult

In [13]:
if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=9999, log_level="debug")

INFO:     Started server process [17632]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9999 (Press CTRL+C to quit)


INFO:     127.0.0.1:55449 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:55454 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:54325 - "POST /generate HTTP/1.1" 200 OK
INFO:     192.168.110.21:50634 - "GET / HTTP/1.1" 404 Not Found
INFO:     192.168.110.21:50634 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:54063 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:54365 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:64243 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:60233 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:60233 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:60233 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:63569 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:63572 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:59246 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:54503 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:54257 - "POST /predict HTTP/1.1" 404 Not Found
I

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [17632]


In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn

ngrok.set_auth_token("3C1UV9DnNJbBB8OO31y1Es6oGIx_23UM9bGLfG4NLq6RDSach")
ngrokTunnel = ngrok.connect(9999)
print("공용 URL", ngrokTunnel.public_url)
nest_asyncio.apply()
uvicorn.run(app, port=9999)